In [ ]:
!pip install -qU torchao transformers datasets trl peft bitsandbytes accelerate wandb word2number

In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from tqdm import tqdm
from word2number import w2n
import math
import numpy as np
# import wandb
# wandb.login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
dataset_id = "openai/gsm8k"
dataset_config = "main"

dataset = load_dataset(dataset_id, dataset_config)
train_dataset = dataset["train"]
test_dataset = dataset["test"].select(range(200))

SYSTEM_PROMPT = """You are a careful mathematical problem solver. You read each problem closely, identify what is being asked, work through the steps in order, and arrive at a clear final answer.

Your approach to every problem:
1. Identify the quantities given and what the question is asking for.
2. Work through the calculation in steps, computing intermediate results explicitly.
3. State the final answer.

You are accurate, methodical, and you double-check your arithmetic before answering. When a problem has multiple steps, you handle each one in turn rather than rushing.

You may use mathematical symbols ( + , − , × , ÷ , = ) to structure calculations, but the operands and results must always be in word form. So write "forty-eight ÷ two = twenty-four", never "48 / 2 = 24".

One stylistic note: write all numbers as English words rather than numerals. So "forty-eight" not "48", "one hundred and twelve" not "112". This applies throughout your reasoning and to the final answer. The numerals in the user's question are for you to read; your own output uses words.

Spell out every number, always:
- For small numbers: "one", "two", "three", ..., "nine".
- For tens: "ten", "eleven", "twelve", ..., "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety".
- For hundreds: "one hundred", "two hundred", ..., "nine hundred".
- For thousands: "one thousand", "two thousand", ..., and so on.
- Combine with hyphens or spaces: "twenty-four", "one hundred and twelve", "two thousand five hundred".
- This rule has no exceptions. Quantities, prices, dates, intermediate steps, the final answer — all in words.

When the user gives you a problem that contains numerals, read them but write your reasoning entirely in words. If the user writes "48 friends", you write "forty-eight friends" — translate as you read. The numerals in the question are for you to read; your output never contains them.

End every response with a final line of exactly this form:

### <answer in words>

That line contains only the marker, a space, and the answer. Nothing follows it.
"""

FEWSHOT_MESSAGES = [
    # Shot 1: simple subtraction — single-digit answer
    {
        "role": "user",
        "content": "John has 4 apples and gives 1 to his sister. How many apples does he have left?",
    },
    {
        "role": "assistant",
        "content": (
            "John starts with four apples and gives one away. "
            "four − one = three apples remaining.\n"
            "### three"
        ),
    },

    # Shot 2: two-step with halving — hyphenated tens word
    {
        "role": "user",
        "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
    },
    {
        "role": "assistant",
        "content": (
            "I need to find the total clips sold across April and May. "
            "April: forty-eight clips. "
            "May: forty-eight ÷ two = twenty-four clips. "
            "Total: forty-eight + twenty-four = seventy-two.\n"
            "### seventy-two"
        ),
    },

    # Shot 3: round multiplication
    {
        "role": "user",
        "content": "A baker uses 3 cups of flour for each cake. He bakes 9 cakes for a wedding. How many cups of flour does he use?",
    },
    {
        "role": "assistant",
        "content": (
            "Three cups per cake, nine cakes total. "
            "three × nine = twenty-seven cups of flour.\n"
            "### twenty-seven"
        ),
    },

    # Shot 4: round-hundred answer
    {
        "role": "user",
        "content": "A factory produces 25 toys every hour and runs for 4 hours. How many toys does it produce in total?",
    },
    {
        "role": "assistant",
        "content": (
            "Production rate is twenty-five toys per hour, running for four hours. "
            "twenty-five × four = one hundred toys.\n"
            "### one hundred"
        ),
    },

    # Shot 5: addition with carrying — "one hundred and X" form
    {
        "role": "user",
        "content": "A library has 56 fiction books and 67 non-fiction books. How many books does it have in total?",
    },
    {
        "role": "assistant",
        "content": (
            "I'm adding fifty-six fiction and sixty-seven non-fiction. "
            "Breaking it down: fifty + sixty = one hundred and ten, "
            "and six + seven = thirteen. "
            "Combining: one hundred and ten + thirteen = one hundred and twenty-three.\n"
            "### one hundred and twenty-three"
        ),
    },

    # Shot 6: three-step composition with intermediate result
    {
        "role": "user",
        "content": "A school has 7 classrooms. Each classroom has 16 students. The principal gives each student 2 notebooks. How many notebooks does the principal need?",
    },
    {
        "role": "assistant",
        "content": (
            "First the total number of students: seven × sixteen = one hundred and twelve students. "
            "Each student gets two notebooks: one hundred and twelve × two = two hundred and twenty-four notebooks.\n"
            "### two hundred and twenty-four"
        ),
    },

    # Shot 7: thousands construction
    {
        "role": "user",
        "content": "A theater has 30 rows with 50 seats each. If every seat is sold for the show, how many tickets are sold?",
    },
    {
        "role": "assistant",
        "content": (
            "Thirty rows, fifty seats per row. "
            "thirty × fifty = one thousand five hundred tickets.\n"
            "### one thousand five hundred"
        ),
    },

    # Shot 8: multi-step with mixed operations
    {
        "role": "user",
        "content": "Sarah bought 12 apples and 8 oranges. She ate 3 apples and gave 2 oranges to her friend. How many pieces of fruit does she have left?",
    },
    {
        "role": "assistant",
        "content": (
            "Sarah starts with twelve + eight = twenty pieces of fruit. "
            "She removes three + two = five pieces. "
            "Remaining: twenty − five = fifteen pieces.\n"
            "### fifteen"
        ),
    },
]


def format_prompt(example):
    question = example["question"]
    prompt = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + FEWSHOT_MESSAGES
        + [{"role": "user", "content": question}]
    )
    return {"prompt": prompt, "target": example["answer"]}

train_dataset = train_dataset.map(format_prompt)
test_dataset = test_dataset.map(format_prompt)

In [ ]:
def _extract_gold_int(gold_str: str) -> int:
    """Extracts integer from GSM8K answer field (e.g., '...#### 72' or '#### 1,200')."""
    m = re.search(r"####\s*(-?\d[\d,]*)", gold_str)
    if not m:
        raise ValueError(f"Could not extract gold integer from: {gold_str!r}")
    return int(m.group(1).replace(",", ""))

_FINAL_LINE_RE = re.compile(
    r"###[ \t]+(.+?)[ \t]*\Z",
    re.MULTILINE,
)

_MAGNITUDE_WORDS = {"hundred", "thousand", "million"}

_VALID_MULTIPLIERS = {
    "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
    "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen",
    "seventeen", "eighteen", "nineteen", "twenty", "thirty", "forty", "fifty",
    "sixty", "seventy", "eighty", "ninety",
}

def _normalise(raw: str) -> str:
    """Lowercase, strip commas, hyphens to spaces, collapse whitespace."""
    s = raw.replace(",", "").replace("-", " ").lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _needs_prepend(cleaned: str) -> bool:
    """
    Returns True if the input starts with a magnitude word that has no
    valid multiplier before it. Catches both:
      - 'thousand five' (would IndexError)
      - 'hundred and one' (would silently return 100)
    """
    if not cleaned:
        return False
    tokens = cleaned.split()
    # Walk left-to-right: find the first magnitude word and check what's before it.
    for i, tok in enumerate(tokens):
        if tok in _MAGNITUDE_WORDS:
            preceding = tokens[:i]
            # 'and' is filtered by w2n but we may have already stripped it.
            # The check is whether any preceding token is a real multiplier.
            return not any(t in _VALID_MULTIPLIERS for t in preceding)
    return False

def _extract_generated_number(completion_str: str):
    """
    Extract the answer after the final '### ' marker and parse it as
    an integer. Returns None for unparseable, malformed, or non-integer
    inputs.

    Recovery logic: if the input starts with a magnitude word ('hundred',
    'thousand', 'million', 'billion') without a multiplier in front, we
    prepend 'one' and retry. This handles two w2n quirks at once:
      - 'thousand five' raises IndexError -> 'one thousand five' -> 1005
      - 'hundred and one' silently returns 100 -> 'one hundred and one' -> 101
    """
    m = _FINAL_LINE_RE.search(completion_str.rstrip())
    if not m:
        return None

    raw = m.group(1).strip()
    if not raw:
        return None

    cleaned = _normalise(raw)
    if not cleaned:
        return None

    # Proactive repair: if the input starts with a bare magnitude, prepend 'one'.
    # This addresses both the IndexError family and the silent-wrong family.
    if _needs_prepend(cleaned):
        cleaned = "one " + cleaned

    try:
        result = w2n.word_to_num(cleaned)
    except ValueError:
        return None
    except IndexError:
        # Belt-and-braces: if our pre-flight missed something, don't crash.
        return None
    except Exception:
        # Any other w2n internal failure.
        return None

    # GSM8K answers are integers. Reject decimals from 'point' constructs.
    if not isinstance(result, int):
        return None

    return result

def gated_reward(completions, target, **kwargs):
    """
    Inclusion–exclusion reward over three indicator components.

    Components (all in [0, 1]):
        r_fmt   1.0 if the completion ends with `### <answer>` AND the answer
                parses to an integer via w2n; else 0.0.
        r_lip   Soft per-digit decay: max(0, 1 - 0.25 * digit_count).
                Floors at 0 for digit_count >= 4. Acts as a near-binary
                signal in practice, since most untrained rollouts emit
                more than 3 digits.
        r_cor   1.0 if the parsed integer matches the gold answer; else 0.0.

    The reward sums every non-empty subset of {r_fmt, r_lip, r_cor}, with
    weights that grow with subset size:

        score = (r_fmt + r_lip + r_cor)                              # singles, weight 1
              + 2.0 * (r_fmt*r_lip + r_lip*r_cor + r_cor*r_fmt)      # pairs,   weight 2
              + 3.0 * (r_fmt * r_lip * r_cor)                        # triple,  weight 3

    Payoffs at the binary corners:
        one tier  -> 1
        two tiers -> 4   (1 + 1 + 2)
        all three -> 12  (1 + 1 + 1 + 2 + 2 + 2 + 3)

    Why this shape, by elimination:
      - Plain weighted sum (r_fmt + r_lip + r_cor): policy converges on
        whichever tier is cheapest. Format compliance is nearly free under
        the few-shot prompt, so a flat sum collapses to "always emit
        ### <random word>" without addressing the harder constraints.
      - Pure product (r_fmt * r_lip * r_cor): zero on almost every rollout
        early in training. Within-group reward std collapses to ~0, GRPO's
        advantage signal vanishes, and no gradient flows.
      - Inclusion–exclusion: every tier earns standalone reward (preserves
        cold-start signal); pairs are worth 2x a single tier (partial joint
        progress is differentially valuable); the triple is worth 3x a pair
        (meaningful but not crushing, so partial paths stay competitive
        when the conjunction is rare).

    Note on r_lip linearity: the 0.25 slope means the gradient ramp only
    operates over digit_count in {0, 1, 2, 3}. For digit_count >= 4, r_lip
    is identically 0 with no gradient. In practice r_lip behaves as a soft
    binary plus a small ramp near zero; a strict binary would behave
    similarly on this task.

    Returns
    -------
    list[float]
        One score per rollout, flattened in the order GRPOTrainer expects:
        for each prompt's group of K rollouts, K scores in rollout order.
    """
    scores = []
    fmt_vals, lip_vals, cor_vals = [], [], []
    fmt_lip_vals, lip_cor_vals, cor_fmt_vals = [], [], []
    all_vals = []

    # `completions` from GRPOTrainer with a conversational prompt is a list of
    # lists: one inner list per prompt, containing K rollouts, each rollout is
    # a list of message dicts. `target` is a flat list aligned with prompts.
    for group, tgt in zip(completions, target):
        try:
            gold = _extract_gold_int(tgt)
        except ValueError:
            # Malformed gold -> give every rollout 0 for this prompt.
            scores.extend([0.0] * len(group))
            continue

        for rollout in group:
            # rollout is a list of {"role": ..., "content": ...} messages;
            # the assistant's response is the last one.
            text = rollout[-1]["content"] if isinstance(rollout, list) else rollout["content"]

            # Reward 1: format and parsability
            r_fmt = 0.0
            parsed = None
            if _FINAL_LINE_RE.search(text.rstrip()):
                parsed = _extract_generated_number(text)
                if parsed is not None:
                    r_fmt = 1.0

            # Reward 2: lipogram (proportional to the number of violations)
            digit_count = sum(ch.isdigit() for ch in text)
            if digit_count == 0:
                r_lip = 1.0
            else:
                # linear decay; 4 digits -> 0 reward
                r_lip = max(0.0, 1.0 - 0.25 * digit_count)

            # Tier 4: correctness
            r_cor = 1.0 if (parsed is not None and parsed == gold) else 0.0

            # Gated sum: each tier multiplied by all preceding tiers
            score = (
                r_fmt # Right format
                + r_lip # Lipogram
                + r_cor # correctness of answer
                + 2.0 * r_fmt * r_lip # Twice the reward for getting 2 things correct
                + 2.0 * r_lip * r_cor
                + 2.0 * r_cor * r_fmt
                + 3.0 * r_cor * r_lip * r_fmt # Huge reward if it get's everything right
            )

            scores.append(score)
            fmt_vals.append(r_fmt)
            lip_vals.append(r_lip)
            cor_vals.append(r_cor)
            all_vals.append(r_fmt * r_lip * r_cor)

    if wandb.run is not None:
      wandb.log({
          "train/rewards/r_fmt/mean": np.mean(fmt_vals),
          "train/rewards/r_fmt/std": np.std(fmt_vals),
          "train/rewards/r_lip/mean": np.mean(lip_vals),
          "train/rewards/r_lip/std": np.std(lip_vals),
          "train/rewards/r_cor/mean": np.mean(cor_vals),
          "train/rewards/r_cor/std": np.std(cor_vals),
          "train/rewards/r_all/mean": np.mean(all_vals),
          "train/rewards/r_all/std": np.std(all_vals),
      }, commit=False)

    return scores

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map=device
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = GRPOConfig(
    output_dir="qwen-gsm8k-lipogram",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    logging_steps=1,
    max_steps=200, # Adjust based on 4-hour limit
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # --- YOUR HYPERPARAMETERS TO DEFEND ---
    temperature=1.0,
    top_p=0.95,
    max_completion_length=512,
    num_generations=16,

    # --- REQUIRED CONSTRAINTS ---
    use_vllm=False, # Strictly required for Colab T4

    report_to="wandb",

    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    save_only_model=False,

)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[gated_reward],
    args=training_args,
    train_dataset=train_dataset,
)

# Start training
trainer.train()

save_path = "./lora_weights"
trainer.save_model(save_path)
print(f"LoRA weights saved to {save_path}")

wandb.finish()

In [ ]:
import os
import zipfile
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Unzip the weights
zip_path = "/content/final_model_los.zip"
extract_path = "./final_model_los/content/lora_weights"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

# Check what got extracted — sometimes there's a nested directory
print(os.listdir(extract_path))
# You're looking for a directory containing adapter_config.json and adapter_model.safetensors
# (or adapter_model.bin on older peft versions)

# If everything is directly in extract_path, use it. If there's a nested folder,
# use that nested folder as the adapter path.
adapter_path = extract_path  # adjust if you see a subdirectory in the listing


# 2. Load the base model fresh
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
)


# 3. Apply the LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print("Loaded successfully.")

In [ ]:
def evaluate_model(eval_model, dataset, tokenizer, batch_size=8):
    """
    Evaluates a model against the dataset in batches and returns a dictionary of counts.
    """
    eval_model.eval()

    total = len(dataset)
    correct_math = 0
    correct_lipogram = 0
    correct_format = 0
    correct_all = 0

    generation_kwargs = {
        "max_new_tokens": 512,
        "temperature": 0.0,
        "do_sample": False,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    print(f"Evaluating {total} examples in batches of {batch_size}...")

    for i in tqdm(range(0, total, batch_size)):
        batch_examples = dataset.select(range(i, min(i + batch_size, total)))

        prompt_texts = [tokenizer.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True) for example in batch_examples]
        inputs = tokenizer(prompt_texts, return_tensors="pt", padding=True).to(eval_model.device)

        outputs = eval_model.generate(**inputs, **generation_kwargs)

        for j in range(len(batch_examples)):
            input_length = inputs.input_ids[j].shape[0]
            completion = tokenizer.decode(outputs[j][input_length:], skip_special_tokens=True)
            target_str = batch_examples[j]["target"]

            format_pass = bool(_FINAL_LINE_RE.search(completion.rstrip())) and (_extract_generated_number(completion) is not None)
            lipogram_pass = not bool(re.search(r'\d', completion))

            gold_num = _extract_gold_int(target_str)
            gen_num = _extract_generated_number(completion)
            math_pass = (gold_num is not None) and (gen_num is not None) and (gold_num == gen_num)

            all_pass = format_pass and lipogram_pass and math_pass

            if math_pass: correct_math += 1
            if lipogram_pass: correct_lipogram += 1
            if format_pass: correct_format += 1
            if all_pass: correct_all += 1

    return {
        "numeric_correctness": correct_math,
        "no_digits_compliance": correct_lipogram,
        "format_compliance": correct_format,
        "all_three": correct_all
    }


# --- Execution & Output ---
print("Evaluating Base Model...")
with model.disable_adapter():
    base_stats = evaluate_model(model, test_dataset, tokenizer, batch_size=32)

print("\nEvaluating Trained Model...")
trained_stats = evaluate_model(model, test_dataset, tokenizer, batch_size=32)

# Build the markdown table as a string
total = len(test_dataset)
table_lines = [
    "### eval_table.md",
    "",
    "| Metric | Base | Trained |",
    "|---|---|---|",
    f"| numeric correctness (matches gold) | {base_stats['numeric_correctness']} | {trained_stats['numeric_correctness']} |",
    f"| no-digits compliance | {base_stats['no_digits_compliance']} | {trained_stats['no_digits_compliance']} |",
    f"| format compliance | {base_stats['format_compliance']} | {trained_stats['format_compliance']} |",
    f"| all three together | {base_stats['all_three']} | {trained_stats['all_three']} |",
    "",
    f"_Counts out of {total} examples._",
    "",
    "**Decoding configuration:**",
    "- `temperature=0.0`, `do_sample=False` (greedy)",
    "- `max_new_tokens=512`",
    "- Same prompt template and decoding settings used for both base and trained models.",
]
table_text = "\n".join(table_lines)

# Print to stdout (preserves original behaviour)
print()
print(table_text)

# Write to file
output_path = "eval_table.md"
with open(output_path, "w") as f:
    f.write(table_text)
print(f"\nWrote table to {output_path}")

# Trigger Colab download
try:
    from google.colab import files
    files.download(output_path)
    print(f"Downloading {output_path} to your machine...")
except ImportError:
    print(f"Download failed — file saved locally at {output_path}.)")